# Mobius export → onnx-world-model inference

This is the complete high-level workflow for `nvidia/Cosmos3-Edge`.

Prerequisites:

```powershell
pip install -e ~/workspace/mobius
pip install -e ~/workspace/onnx-world-model
```

The FP32 package is about 23 GB. CPU export and inference are supported but slow; use BF16 and CUDA in a CUDA environment.

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image
from onnx_world_model import WorldModel

EXPORT_DIR = Path("artifacts/cosmos3-edge-f32")

## 1. Export with Mobius

One command builds every component, applies the published weights, copies runtime assets, and writes `pipeline.json`.

In [ ]:
!mobius build --model nvidia/Cosmos3-Edge {EXPORT_DIR} --features world-model --dtype f32

## 2. Run with onnx-world-model

One high-level call loads the Mobius package and generates an image.

In [ ]:
result = WorldModel.from_pretrained(EXPORT_DIR, providers=["cpu"]).image.generate("A photorealistic orange cat sitting on a windowsill", height=256, width=256, guidance_scale=5.0, num_inference_steps=50, seed=42)

In [ ]:
pixels = np.clip((result.images[0].transpose(1, 2, 0) + 1.0) * 127.5, 0, 255).astype(np.uint8)
Image.fromarray(pixels)

The same loaded object also exposes `model.text.generate(...)`, `model.video.generate(...)`, and `model.action.generate(...)`. For image-to-video, pass `image=` to `model.video.generate(...)`.